# L1c · R11 to 8-bit (INT8): the before-and-after model, for the website and the field (CPU, no GPU)

Task **L1c**, the website's work-list item 5 (`Progress_Tracker.md`, "Needs you now"). Built 17 Nov 2026. Goal **O4** (a model small and fast enough for a laptop CPU or a browser).

**Why:**
- The website's "Try it" page runs R9 in the visitor's browser: our after-photo-only U-Net, exported to 8-bit by L1b (33 MB, 1.43 times the speed of the full-precision export, a 0.0003 drop in destroyed IoU).
- R11, the U-Net that reads the after-photo and the before-photo together, is our better map on mIoU (0.5742 against 0.5338 on the clean pairs, without TTA). It has never been exported.
- This notebook does for R11 exactly what L1b did for R9.

**What is the same as L1b** (`l1b-static-int8-retry.ipynb`, kept as it ran):
- the ONNX export (opset 17, a fixed 1024 × 1024 input);
- static INT8 in QDQ format, per-channel weights, activation ranges from 100 training images (never validation);
- the two versions with a few convolutions kept in full precision (FP32):
  - **keep 1+1:** the encoder's stem and the segmentation head;
  - **keep 1+3:** the same, plus the two convolutions of the last decoder block;
- accuracy without TTA on all 645 validation pairs;
- speed as the median of 3 timed runs per image on 50 images, after a warm-up.

**What changes for R11:**
1. **Six input channels:** channels 0–2 are the after-photo and 3–5 the before-photo, each with the ImageNet normalisation, as R11 was trained (E7's and the gallery export's `unet_input`). The before-photo is resized to the after-photo's size if they differ, as there.
2. **The pairs keep their before-photo** (R11's own Cell 4, `pre_image_path`).
3. **The full-precision accuracy is measured here,** since no L1 exists for R11:
   - ONNX FP32 on all 645 must reproduce E5 step 2's R11 without TTA, 0.5811 mIoU / 0.2911 destroyed IoU, within 0.001;
   - PyTorch and ONNX FP32 must give the same class on at least 99.99% of pixels on 3 images.
4. **Scores on the 605 clean pairs too,** without the 40 pairs that share images with training (E5 step 1; their names are in Cell 2).
5. **`r11_fp32.onnx` is kept** in the outputs: it's new, and the website could use it on WebGPU later.

**The rule, fixed before the run (as L1b's):** a static INT8 version is usable if its destroyed IoU drops by less than 0.01 against ONNX FP32 on the 645 pairs. The fastest usable version becomes R11's edge model. If neither passes, ONNX FP32 stays R11's edge model. The drop on the 605 clean pairs is reported beside it.

**Expectation, fixed before the run:**
- ONNX FP32 reproduces E5's numbers.
- Both INT8 versions pass, with drops near L1b's 0.0003, since only the first layer differs from R9.
- The files are about 33 MB. Each runs about 1.4 times faster than FP32, a little slower than R9's, because the stem reads 6 channels.

**How to run on Kaggle** (account 1 or 2: R11's checkpoint is private)
1. Settings: **CPU** (no accelerator), internet **on**, secret `HF_TOKEN` (AbrarAlam, read access).
2. Input: `abrarmohammedtanzim/disasterm3-mirror`.
3. Run with `SMOKE_TEST = True` first (20 validation pairs, 10 calibration and 5 latency images; about 20 minutes).
4. Then set it to `False`, and use Save & Run All (about 2–2.5 hours, estimated).

**Outputs** in `/kaggle/working/L1c/`:
- `int8_r11.csv`, `README.md`, `config.json`, `side_by_side.png`;
- `r11_fp32.onnx` and the two INT8 `.onnx` files.

The README gives each file's exact size in bytes, which the website's `modelConfig.ts` needs.

**For the website afterwards** (not done here): point `FIELD_MODEL` in `T3_Website/src/core/modelConfig.ts` at the chosen file, set its size, and set `images: ['after', 'before']`. That order is R11's channel order, after-photo first. Try it then asks for two photos.

In [1]:
# ── Cell 1: Install the export and quantisation tools (same as L1) ─────────
import importlib.metadata as _md
import subprocess
import sys

PINNED = {"segmentation-models-pytorch": "0.3.4"}     # same version as the D4 notebooks (checkpoint key layout)
EXTRA = ["onnx", "onnxruntime", "onnxscript"]           # onnxscript: needed by the newer torch ONNX exporter

def _version(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return None

missing = [f"{p}=={v}" for p, v in PINNED.items() if _version(p) != v] + [p for p in EXTRA if _version(p) is None]
if missing:
    print("⏳ Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import onnx
import onnxruntime as ort
import torch
print(f"✓ torch {torch.__version__}, onnx {onnx.__version__}, onnxruntime {ort.__version__}, "
      f"python {sys.version.split()[0]}")

⏳ Installing: segmentation-models-pytorch==0.3.4, onnxruntime, onnxscript
✓ torch 2.11.0+cpu, onnx 1.23.2, onnxruntime 1.32.0, python 3.13.18


In [2]:
# ── Cell 2: Configuration, checkpoint and the machine ─────────────────────
import glob
import json
import os
import platform
import random
import time
from pathlib import Path

import numpy as np
import psutil
from huggingface_hub import HfApi, hf_hub_download

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"
OUT_DIR = Path("/kaggle/working/L1c")

NUM_CLASSES = 4
SMOKE_TEST = False             # True = 20 validation, 10 calibration and 5 latency images
MODEL_NAME, HF_USER = "R11", "AbrarAlam"
CKPT_PATH = None               # set a path to skip the search, e.g. "/kaggle/input/my-ckpts/R11/final_model.pth"
IN_CHANNELS = 6                # channels 0-2 the after-photo, 3-5 the before-photo, as R11 was trained
INPUT_SIZE = 1024              # every validation image is 1024 x 1024 (checked); the ONNX input is fixed
OPSET = 17
N_CALIBRATION = 100            # training pairs for static INT8 calibration
N_LATENCY = 50                 # validation pairs for the latency runs
LATENCY_REPEATS = 3            # timed runs per image after a warm-up; the median is kept
N_AGREEMENT = 3                # validation pairs on which PyTorch and ONNX FP32 must agree
STATIC_KEEP_FP32 = {"INT8 static, keep 1+1 FP32": (1, 1), "INT8 static, keep 1+3 FP32": (1, 3)}   # as L1b
E5_R11_STD = {"miou": 0.5811, "destroyed_iou": 0.2911}   # E5 step 2: R11 without TTA, all 645 validation pairs
MAX_DROP = 0.01                # INT8 counts as usable if destroyed IoU drops by less than this (L1b's rule)
REPRO_TOL = 0.001              # ONNX FP32 must reproduce E5's numbers within this
MIN_AGREEMENT = 0.9999         # PyTorch against ONNX FP32: the share of pixels with the same class
# E5 step 1's 40 validation pairs that share images with training (val_train_leaks_hash.csv, counted True)
LEAKED = {f"socal_fire_{n:08d}_post_disaster.png" for n in
          (81, 188, 218, 220, 222, 235, 301, 341, 348, 396, 436, 455, 497, 504, 614, 1188)} | \
         {f"woolsey_fire_{n:08d}_post_disaster.png" for n in
          (9, 30, 32, 137, 272, 399, 435, 474, 498, 514, 529, 573, 576, 589, 623, 632, 634, 653, 679, 703, 722, 758,
           783, 810)}
assert len(LEAKED) == 40


def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")


def find_checkpoint(name):
    """Order: CKPT_PATH, then a /kaggle/input folder named R11 / d4_R11 / R11-results, then Hugging Face.
    Returns the path and where it came from (the repository and commit, when from Hugging Face)."""
    if CKPT_PATH:
        return CKPT_PATH, CKPT_PATH
    wanted = {name.lower(), f"d4_{name}".lower(), f"{name}-results".lower()}
    hits = [p for p in glob.glob("/kaggle/input/**/*.pth", recursive=True)
            if any(part.lower() in wanted for part in Path(p).parts)]
    hits.sort(key=lambda p: Path(p).name != "final_model.pth")   # prefer final_model.pth
    if hits:
        return hits[0], hits[0]
    repo, errors = f"{HF_USER}/disasterm3-d4-{name}", []
    sha = HfApi().model_info(repo, token=hf_token()).sha
    for filename in ("final_model.pth", "last.pth"):   # last.pth holds the same epoch-40 weights
        try:
            return hf_hub_download(repo, filename, revision=sha, token=hf_token()), f"{repo}@{sha[:8]}"
        except Exception as e:
            errors.append(f"{filename}: {type(e).__name__}")
    raise FileNotFoundError(f"No checkpoint for {name}: nothing under /kaggle/input and {repo} gave {errors}.")


def cpu_name():
    try:
        with open("/proc/cpuinfo") as f:
            return next(line.split(":", 1)[1].strip() for line in f if line.startswith("model name"))
    except (OSError, StopIteration):
        return platform.processor() or platform.machine()


CKPT, CKPT_SOURCE = find_checkpoint(MODEL_NAME)
OUT_DIR.mkdir(parents=True, exist_ok=True)
THREADS = os.cpu_count()
torch.set_num_threads(THREADS)
MACHINE = {"cpu": cpu_name(), "threads": THREADS, "ram_gb": round(psutil.virtual_memory().total / 1e9, 1),
           "os": platform.platform(), "torch_threads": torch.get_num_threads()}
print(f"✓ L1c R11 to INT8{'  [SMOKE TEST]' if SMOKE_TEST else ''}: {MODEL_NAME} from {CKPT_SOURCE}")
print(f"  Machine: {MACHINE}")

final_model.pth: reconstructing file:   0%|          |  0.00B /  130MB            

final_model.pth: downloading bytes:           |  0.00B            

✓ L1c R11 to INT8: R11 from AbrarAlam/disasterm3-d4-R11@31974dac
  Machine: {'cpu': 'Intel(R) Xeon(R) CPU @ 2.20GHz', 'threads': 4, 'ram_gb': 33.7, 'os': 'Linux-6.18.55+-x86_64-with-glibc2.39', 'torch_threads': 4}


In [3]:
# ── Cell 3: Load manifest and build the 4-class masks (R11's own Cell 4: the D4 code plus the before-photo) ──
import cv2
import numpy as np
import os
from collections import Counter, defaultdict
from pathlib import Path
import json

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from manifest")

# ── Filter: ONLY Building Damage Assessment segmentation entries ──
bda_entries = [
    e for e in raw_data
    if e.get("task") == "Referring Expression Segmentation"
    and e.get("cls_description") == "Building Damage Assessment"
]
print(f"Building Damage Assessment entries: {len(bda_entries):,}")

# ── Path resolution (same logic as before) ──
def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

# ── Group entries by underlying post-disaster image ──
by_image = defaultdict(dict)  # post_image_path -> {folder_type: mask_path}

for e in bda_entries:
    post_img = e.get("post_image_path", "")
    if e.get("image_type") != "Optical":   # skip SAR duplicates, keep optical only
        continue
    mask_rel = e.get("ground_truth", "")
    folder = Path(mask_rel.replace("\\", "/")).parent.name  # e.g. train_building_intact_mask
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["post_image_path"] = post_img
        by_image[post_img]["pre_image_path"] = e.get("pre_image_path")   # R11: the before-image

print(f"Unique images with at least one mask: {len(by_image):,}")

# ── Build combined (image, mask) pairs ──
FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,   # use 2 here instead if you want to collapse into "damaged"
}

# ── FIX: Create temporary directory on Kaggle hard drive for masks ──
os.makedirs("/tmp/combined_masks", exist_ok=True)

pairs = []
skipped_img = 0

for post_img, mask_dict in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path:
        skipped_img += 1
        continue

    combined_mask = None
    for folder, class_id in FOLDER_TO_CLASS.items():
        mask_path = mask_dict.get(folder)
        if mask_path is None:
            continue
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if combined_mask is None:
            combined_mask = np.zeros_like(m, dtype=np.uint8)
        combined_mask[m > 0] = class_id   # later classes overwrite earlier if they overlap

    if combined_mask is None:
        continue  # no valid masks found for this image at all

    # ── FIX: Save mask to hard drive instead of RAM ──
    mask_filename = Path(img_path).name
    save_path = f"/tmp/combined_masks/{mask_filename}"
    cv2.imwrite(save_path, combined_mask)

    pairs.append({
        "image_path": img_path,
        "mask_path": save_path,   # Store the string path, NOT the array!
        "pre_image_path": resolve_path(mask_dict.get("pre_image_path")),   # R11: None if not found
    })

print(f"\n✓ Built {len(pairs):,} valid (image, mask) pairs")
print(f"  Skipped (image not found): {skipped_img:,}")

# Sanity check
if pairs:
    print(f"\n  Sample: {pairs[0]['image_path']}")
    print(f"  Mask saved at: {pairs[0]['mask_path']}")

Loaded 92,968 entries from manifest
Building Damage Assessment entries: 14,531
Unique images with at least one mask: 6,443

✓ Built 6,443 valid (image, mask) pairs
  Skipped (image not found): 0

  Sample: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_images/train_images/bata_explosion_post_0.png
  Mask saved at: /tmp/combined_masks/bata_explosion_post_0.png


In [4]:
# ── Cell 4: Validation split (identical to every D4 run), calibration pairs and the 6-channel preprocessing ──
from sklearn.model_selection import train_test_split

missing_pre = sum(p["pre_image_path"] is None for p in pairs)
assert missing_pre == 0, f"{missing_pre} pairs have no before-photo; R11 needs every one"
train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)
calib_pairs = random.Random(0).sample(train_pairs, N_CALIBRATION)   # training pairs only, never validation
if SMOKE_TEST:
    val_pairs, calib_pairs = val_pairs[:20], calib_pairs[:10]
latency_pairs = val_pairs[:5 if SMOKE_TEST else N_LATENCY]
clean_val = [p for p in val_pairs if Path(p["image_path"]).name not in LEAKED]

# The D4 notebooks' A.Normalize (ImageNet mean and std, max_pixel_value 255), without albumentations
MEAN = np.array([0.485, 0.456, 0.406], np.float32) * 255
STD = np.array([0.229, 0.224, 0.225], np.float32) * 255


def stack(post, pre):
    """After-photo and before-photo (RGB uint8, H x W x 3) -> float32 1 x 6 x H x W: channels 0-2 the after-photo,
    3-5 the before-photo, normalised exactly as R11 was trained. The before-photo is resized to the after-photo's
    size if they differ, as in E7 and the gallery export."""
    if pre.shape != post.shape:
        pre = cv2.resize(pre, (post.shape[1], post.shape[0]), interpolation=cv2.INTER_LINEAR)
    x = np.concatenate([(post.astype(np.float32) - MEAN) / STD, (pre.astype(np.float32) - MEAN) / STD], 2)
    return np.ascontiguousarray(x.transpose(2, 0, 1)[None])


def preprocess(pair):
    """One pair's files -> the model's 6-channel input."""
    rgb = lambda path: cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    return stack(rgb(pair["image_path"]), rgb(pair["pre_image_path"]))


# Self-checks of the stacking: channel order, normalisation, and the before-photo's resize
_post = np.full((4, 4, 3), 255, np.uint8)
_pre = np.zeros((2, 2, 3), np.uint8)
_x = stack(_post, _pre)
assert _x.shape == (1, 6, 4, 4) and _x.dtype == np.float32
assert np.allclose(_x[0, :3], ((255 - MEAN) / STD)[:, None, None]) and np.allclose(_x[0, 3:], (-MEAN / STD)[:, None, None])

sizes = Counter(cv2.imread(p["mask_path"], cv2.IMREAD_GRAYSCALE).shape for p in val_pairs)
assert set(sizes) == {(INPUT_SIZE, INPUT_SIZE)}, f"The fixed ONNX input needs {INPUT_SIZE} px images, found {sizes}"
if not SMOKE_TEST:
    assert len(val_pairs) == 645 and len(clean_val) == 605, f"{len(val_pairs)} validation pairs, {len(clean_val)} clean"
print(f"✓ Validation pairs: {len(val_pairs)} ({len(clean_val)} clean); calibration: {len(calib_pairs)} training pairs; "
      f"latency: {len(latency_pairs)} pairs; stacking self-checks passed")

✓ Validation pairs: 645 (605 clean); calibration: 100 training pairs; latency: 50 pairs; stacking self-checks passed


In [5]:
# ── Cell 5: Export to ONNX FP32, check it against PyTorch, then static INT8 with a few convolutions kept in FP32 ──
import segmentation_models_pytorch as smp
from onnxruntime.quantization import CalibrationDataReader, QuantFormat, QuantType, quantize_static
from onnxruntime.quantization.shape_inference import quant_pre_process


def load_pytorch():
    """D4 checkpoints: U-Net + ResNet-50, 4 classes; R11 reads 6 channels. Accepts a state dict or a resume checkpoint."""
    state = torch.load(CKPT, map_location="cpu", weights_only=False)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model = smp.Unet(encoder_name="resnet50", encoder_weights=None, in_channels=IN_CHANNELS, classes=NUM_CLASSES)
    model.load_state_dict(state)
    return model.eval()


PATHS = {"ONNX FP32": OUT_DIR / "r11_fp32.onnx"}
prepared = OUT_DIR / "r11_fp32_prepared.onnx"     # shape-inferred and optimised copy that the quantiser starts from

t0 = time.time()
model = load_pytorch()
dummy = torch.zeros(1, IN_CHANNELS, INPUT_SIZE, INPUT_SIZE)
export_args = dict(input_names=["image"], output_names=["logits"], opset_version=OPSET)
try:
    torch.onnx.export(model, (dummy,), str(PATHS["ONNX FP32"]), dynamo=False, **export_args)
except Exception as e:                            # newer torch may drop the classic exporter
    print(f"Classic exporter failed ({type(e).__name__}: {str(e)[:100]}); using the dynamo exporter")
    torch.onnx.export(model, (dummy,), str(PATHS["ONNX FP32"]), dynamo=True, **export_args)
onnx.checker.check_model(str(PATHS["ONNX FP32"]))

# Check: PyTorch and ONNX FP32 give the same class on almost every pixel of a few validation pairs
session = ort.InferenceSession(str(PATHS["ONNX FP32"]), providers=["CPUExecutionProvider"])
agreements = []
with torch.no_grad():
    for p in val_pairs[:N_AGREEMENT]:
        x = preprocess(p)
        a = model(torch.from_numpy(x))[0].argmax(0).numpy()
        b = session.run(None, {"image": x})[0][0].argmax(0)
        agreements.append(float((a == b).mean()))
del session, model
AGREEMENT = min(agreements)
assert AGREEMENT >= MIN_AGREEMENT, f"PyTorch and ONNX FP32 agree on only {AGREEMENT:.6f} of pixels"
print(f"✓ ONNX FP32 matches PyTorch: the same class on at least {AGREEMENT:.6f} of pixels ({N_AGREEMENT} pairs)")

quant_pre_process(str(PATHS["ONNX FP32"]), str(prepared))
convs = [n.name for n in onnx.load(str(prepared)).graph.node if n.op_type == "Conv"]
print(f"{len(convs)} convolutions; first {convs[0]!r}, last {convs[-1]!r}")


class TrainingPairs(CalibrationDataReader):
    """Feeds the calibration pairs one by one, so static INT8 learns realistic activation ranges."""
    def __init__(self, pairs):
        self.items = iter(pairs)

    def get_next(self):
        p = next(self.items, None)
        return None if p is None else {"image": preprocess(p)}


# As L1b: don't set extra_options={"CalibMaxIntermediateOutputs": ...}; in onnxruntime 1.31 it clears the collected
# data, and calibration then fails with "No data is collected" (the first L1 run, 18 Oct).
EXCLUDED = {}
for kind, (n_first, n_last) in STATIC_KEEP_FP32.items():
    EXCLUDED[kind] = convs[:n_first] + convs[-n_last:]
    PATHS[kind] = OUT_DIR / f"r11_int8_static_keep{n_first}_{n_last}.onnx"
    quantize_static(str(prepared), str(PATHS[kind]), TrainingPairs(calib_pairs),
                    quant_format=QuantFormat.QDQ, per_channel=True,
                    weight_type=QuantType.QInt8, activation_type=QuantType.QUInt8,
                    nodes_to_exclude=EXCLUDED[kind])
    print(f"  {kind}: FP32 kept for {EXCLUDED[kind]}")

FILE_BYTES = {k: os.path.getsize(p) for k, p in PATHS.items()}
print(f"✓ Export and quantisation done in {(time.time() - t0) / 60:.1f} min")
for k, b in FILE_BYTES.items():
    print(f"  {k:28s} {b / 1e6:7.1f} MB ({b:,} bytes)")

/tmp/ipykernel_16/1834489067.py:25: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, (dummy,), str(PATHS["ONNX FP32"]), dynamo=False, **export_args)
/usr/local/lib/python3.13/dist-packages/segmentation_models_pytorch/base/model.py:17: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if h % output_stride != 0 or w % output_stride != 0:


✓ ONNX FP32 matches PyTorch: the same class on at least 1.000000 of pixels (3 pairs)
64 convolutions; first '/encoder/conv1/Conv', last '/segmentation_head/segmentation_head.0/Conv'
  INT8 static, keep 1+1 FP32: FP32 kept for ['/encoder/conv1/Conv', '/segmentation_head/segmentation_head.0/Conv']
  INT8 static, keep 1+3 FP32: FP32 kept for ['/encoder/conv1/Conv', '/decoder/blocks.4/conv1/conv1.0/Conv', '/decoder/blocks.4/conv2/conv2.0/Conv', '/segmentation_head/segmentation_head.0/Conv']
✓ Export and quantisation done in 16.5 min
  ONNX FP32                      130.0 MB (130,048,299 bytes)
  INT8 static, keep 1+1 FP32      33.1 MB (33,145,715 bytes)
  INT8 static, keep 1+3 FP32      33.2 MB (33,163,619 bytes)


## Measure: accuracy of ONNX FP32 and the two static versions, then their speed
- **Accuracy:** the standard prediction (no TTA) and the dataset-level confusion matrix, as in L1b and the D4 notebooks, on all 645 validation pairs and on the 605 clean ones. ONNX FP32 is measured here (R11 has no L1) and must reproduce E5 step 2's R11 without TTA. The drop is measured against it.
- **Speed:** all three versions timed in this session: a warm-up, then 3 timed runs per pair on the first 50 validation pairs; the median per pair is kept.

In [6]:
# ── Cell 6: Accuracy of ONNX FP32 and each static INT8 version on the validation pairs (all, and clean) ──
import gc
import pandas as pd
from tqdm.auto import tqdm

VERSIONS = ["ONNX FP32", *STATIC_KEEP_FP32]


def make_runner(kind):
    """6-channel input (1 x 6 x H x W) -> logits array, for one ONNX file."""
    options = ort.SessionOptions()
    options.intra_op_num_threads = THREADS
    session = ort.InferenceSession(str(PATHS[kind]), options, providers=["CPUExecutionProvider"])
    return lambda x: session.run(None, {"image": x})[0]


def confusion(gt, pred):
    idx = gt.astype(np.int64).ravel() * NUM_CLASSES + pred.astype(np.int64).ravel()
    return np.bincount(idx, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)


def iou_from(cm):
    tp = np.diag(cm).astype(np.float64)
    return tp / (cm.sum(0) + cm.sum(1) - tp + 1e-8)


assert np.allclose(iou_from(confusion(np.array([0, 1, 2, 3]), np.array([0, 1, 2, 3]))), 1.0)

clean_names = {Path(p["image_path"]).name for p in clean_val}
examples, accuracy = {}, {}
t0 = time.time()
for kind in VERSIONS:
    run = make_runner(kind)
    cm_all = np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64)
    cm_clean = np.zeros_like(cm_all)
    for i, p in enumerate(tqdm(val_pairs, desc=kind)):
        pred = run(preprocess(p))[0].argmax(0).astype(np.uint8)
        cm = confusion(cv2.imread(p["mask_path"], cv2.IMREAD_GRAYSCALE), pred)
        cm_all += cm
        if Path(p["image_path"]).name in clean_names:
            cm_clean += cm
        if i == 0:
            examples[kind] = pred
    iou_all, iou_clean = iou_from(cm_all), iou_from(cm_clean)
    accuracy[kind] = {"miou": float(iou_all.mean()), "destroyed_iou": float(iou_all[3]),
                      "miou_clean": float(iou_clean.mean()), "destroyed_iou_clean": float(iou_clean[3])}
    print(f"  {kind:28s} all {len(val_pairs)}: mIoU {iou_all.mean():.4f} | destroyed IoU {iou_all[3]:.4f}   "
          f"clean {len(clean_val)}: mIoU {iou_clean.mean():.4f} | destroyed IoU {iou_clean[3]:.4f}   "
          f"({(time.time() - t0) / 60:.0f} min so far)")
    del run
    gc.collect()

# Check: ONNX FP32 reproduces E5 step 2's R11 without TTA (only meaningful on all 645 pairs)
fp32 = accuracy["ONNX FP32"]
REPRODUCED = all(abs(fp32[k] - E5_R11_STD[k]) <= REPRO_TOL for k in E5_R11_STD)
if not SMOKE_TEST and not REPRODUCED:
    raise RuntimeError(f"ONNX FP32 gives {fp32['miou']:.4f} / {fp32['destroyed_iou']:.4f}, but E5 step 2 measured "
                       f"{E5_R11_STD['miou']} / {E5_R11_STD['destroyed_iou']}: the export or the inputs differ. Stop here.")
print(f"✓ ONNX FP32 {'reproduces' if REPRODUCED else 'differs from'} E5 step 2's R11 "
      f"({E5_R11_STD['miou']} / {E5_R11_STD['destroyed_iou']}){'  [SMOKE TEST: 20 pairs, not comparable]' if SMOKE_TEST else ''}")

ONNX FP32:   0%|          | 0/645 [00:00<?, ?it/s]

  ONNX FP32                    all 645: mIoU 0.5811 | destroyed IoU 0.2911   clean 605: mIoU 0.5742 | destroyed IoU 0.2641   (31 min so far)


INT8 static, keep 1+1 FP32:   0%|          | 0/645 [00:00<?, ?it/s]

  INT8 static, keep 1+1 FP32   all 645: mIoU 0.5801 | destroyed IoU 0.2897   clean 605: mIoU 0.5734 | destroyed IoU 0.2630   (53 min so far)


INT8 static, keep 1+3 FP32:   0%|          | 0/645 [00:00<?, ?it/s]

  INT8 static, keep 1+3 FP32   all 645: mIoU 0.5801 | destroyed IoU 0.2897   clean 605: mIoU 0.5733 | destroyed IoU 0.2630   (75 min so far)
✓ ONNX FP32 reproduces E5 step 2's R11 (0.5811 / 0.2911)


In [7]:
# ── Cell 7: Speed of ONNX FP32 and both static versions (median of 3 timed runs per pair, after a warm-up) ──
inputs = [preprocess(p) for p in latency_pairs]
latency = {}
for kind in VERSIONS:
    run = make_runner(kind)
    run(inputs[0])                                   # warm-up (first call allocates and optimises)
    per_image = []
    for x in tqdm(inputs, desc=kind):
        times = []
        for _ in range(LATENCY_REPEATS):
            t = time.perf_counter()
            run(x)
            times.append(time.perf_counter() - t)
        per_image.append(1000 * float(np.median(times)))
    latency[kind] = {"ms_median": float(np.median(per_image)), "ms_p95": float(np.percentile(per_image, 95))}
    print(f"  {kind:28s} {latency[kind]['ms_median']:8.0f} ms per image (median), "
          f"p95 {latency[kind]['ms_p95']:.0f} ms")
    del run
    gc.collect()

ONNX FP32:   0%|          | 0/50 [00:00<?, ?it/s]

  ONNX FP32                        2364 ms per image (median), p95 2382 ms


INT8 static, keep 1+1 FP32:   0%|          | 0/50 [00:00<?, ?it/s]

  INT8 static, keep 1+1 FP32       1835 ms per image (median), p95 1857 ms


INT8 static, keep 1+3 FP32:   0%|          | 0/50 [00:00<?, ?it/s]

  INT8 static, keep 1+3 FP32       1751 ms per image (median), p95 1770 ms


In [8]:
# ── Cell 8: The table, the verdict, README, config and a side-by-side picture ──
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

STATIC = list(STATIC_KEEP_FP32)
base_ms, fp32 = latency["ONNX FP32"]["ms_median"], accuracy["ONNX FP32"]
rows = []
for kind in VERSIONS:
    a, l = accuracy[kind], latency[kind]
    drop = fp32["destroyed_iou"] - a["destroyed_iou"]
    rows.append({"variant": kind, "file_mb": round(FILE_BYTES[kind] / 1e6, 1), "bytes": FILE_BYTES[kind],
                 "ms_per_image": round(l["ms_median"]), "ms_p95": round(l["ms_p95"]),
                 "speedup_vs_onnx_fp32": round(base_ms / l["ms_median"], 2),
                 "miou": round(a["miou"], 4), "destroyed_iou": round(a["destroyed_iou"], 4),
                 "destroyed_drop": round(drop, 4),
                 "miou_clean": round(a["miou_clean"], 4), "destroyed_iou_clean": round(a["destroyed_iou_clean"], 4),
                 "destroyed_drop_clean": round(fp32["destroyed_iou_clean"] - a["destroyed_iou_clean"], 4),
                 "int8_usable": "" if kind == "ONNX FP32" else ("yes" if drop < MAX_DROP else "no"),
                 "kept_in_fp32": "everything" if kind == "ONNX FP32" else ", ".join(EXCLUDED[kind])})
table = pd.DataFrame(rows)
table.to_csv(OUT_DIR / "int8_r11.csv", index=False)
print(table.drop(columns=["kept_in_fp32"]).to_string(index=False))

usable = table[table["int8_usable"] == "yes"].sort_values("ms_per_image")
if len(usable):
    best = usable.iloc[0]
    verdict = (f"R11's edge model: {best['variant']} ({Path(PATHS[best['variant']]).name}, {best['bytes']:,} bytes), "
               f"{best['speedup_vs_onnx_fp32']}x the speed of ONNX FP32, destroyed IoU drop {best['destroyed_drop']:.4f} "
               f"(limit {MAX_DROP}; {best['destroyed_drop_clean']:.4f} on the clean pairs).")
else:
    verdict = f"Neither static version passes (limit {MAX_DROP}): ONNX FP32 stays R11's edge model."
print(f"\n{verdict}")

# The first validation pair: ground truth and each version (0 background, 1 intact, 2 damaged, 3 destroyed)
colours = np.array([[0, 0, 0], [60, 180, 75], [255, 200, 0], [230, 25, 75]], np.uint8)
panels = [("Ground truth", cv2.imread(val_pairs[0]["mask_path"], cv2.IMREAD_GRAYSCALE))] + \
         [(kind, examples[kind]) for kind in VERSIONS]
fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 4.4))
for ax, (title, m) in zip(axes, panels):
    ax.imshow(colours[m])
    ax.set_title(title)
    ax.axis("off")
fig.suptitle(Path(val_pairs[0]["image_path"]).name)
fig.tight_layout()
fig.savefig(OUT_DIR / "side_by_side.png", dpi=90)
plt.close(fig)


def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"


prepared.unlink(missing_ok=True)     # an intermediate copy
config = {"task": "L1c", "smoke_test": SMOKE_TEST, "model": MODEL_NAME, "checkpoint": CKPT_SOURCE, "machine": MACHINE,
          "in_channels": IN_CHANNELS, "channel_order": "after-photo (0-2), before-photo (3-5)",
          "input_size": INPUT_SIZE, "opset": OPSET, "calibration_pairs": len(calib_pairs),
          "validation_pairs": len(val_pairs), "clean_pairs": len(clean_val), "latency_pairs": len(latency_pairs),
          "latency_repeats": LATENCY_REPEATS, "kept_in_fp32": EXCLUDED, "e5_r11_std": E5_R11_STD,
          "reproduces_e5": REPRODUCED, "pytorch_onnx_agreement": AGREEMENT, "file_bytes": FILE_BYTES,
          "onnxruntime": ort.__version__, "torch": torch.__version__}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    "# L1c: R11 to 8-bit (INT8)", "",
    f"Machine: {MACHINE['cpu']}, {MACHINE['threads']} threads, {MACHINE['ram_gb']} GB RAM ({MACHINE['os']}). "
    f"onnxruntime {ort.__version__}, torch {torch.__version__}. Checkpoint: {CKPT_SOURCE}.", "",
    "L1b's recipe for R11, the U-Net that reads the after-photo (channels 0-2) and the before-photo (3-5): QDQ, "
    "per-channel weights, ranges from 100 training pairs, a few convolutions kept in FP32. Accuracy without TTA on "
    f"{len(val_pairs)} validation pairs ({len(clean_val)} clean){' (SMOKE TEST)' if SMOKE_TEST else ''}; speed as the "
    f"median of {LATENCY_REPEATS} timed runs per image on {len(latency_pairs)} pairs, all versions in one session.", "",
    f"Checks: PyTorch and ONNX FP32 give the same class on at least {AGREEMENT:.6f} of pixels ({N_AGREEMENT} pairs); "
    f"ONNX FP32 {'reproduces' if REPRODUCED else 'does NOT reproduce'} E5 step 2's R11 without TTA "
    f"({E5_R11_STD['miou']} / {E5_R11_STD['destroyed_iou']}, within {REPRO_TOL}).", "",
    as_markdown(table.drop(columns=["kept_in_fp32"])), "",
    f"**Verdict:** {verdict}", "",
    "Convolutions kept in FP32:", "",
    *[f"- {kind}: {', '.join(names)}" for kind, names in EXCLUDED.items()], "",
    "For the website: point `FIELD_MODEL` in `T3_Website/src/core/modelConfig.ts` at the chosen file, with its exact "
    "`bytes` from the table, and set `images: ['after', 'before']` (R11's channel order).", "",
    "![Side by side](side_by_side.png)", "",
    "Files: `int8_r11.csv`, `r11_fp32.onnx`, the two static INT8 `.onnx` files, `config.json`.",
]), encoding="utf-8")
print(f"✓ Wrote int8_r11.csv, side_by_side.png, README.md and config.json to {OUT_DIR}")

                   variant  file_mb     bytes  ms_per_image  ms_p95  speedup_vs_onnx_fp32   miou  destroyed_iou  destroyed_drop  miou_clean  destroyed_iou_clean  destroyed_drop_clean int8_usable
                 ONNX FP32    130.0 130048299          2364    2382                  1.00 0.5811         0.2911          0.0000      0.5742               0.2641                0.0000            
INT8 static, keep 1+1 FP32     33.1  33145715          1835    1857                  1.29 0.5801         0.2897          0.0014      0.5734               0.2630                0.0011         yes
INT8 static, keep 1+3 FP32     33.2  33163619          1751    1770                  1.35 0.5801         0.2897          0.0014      0.5733               0.2630                0.0011         yes

R11's edge model: INT8 static, keep 1+3 FP32 (r11_int8_static_keep1_3.onnx, 33,163,619 bytes), 1.35x the speed of ONNX FP32, destroyed IoU drop 0.0014 (limit 0.01; 0.0011 on the clean pairs).
✓ Wrote int8_r11.csv, side_